# Figure 2: Representative reach-date filtering example

This notebook uses the established reach 66405900761 example from 2024-03-10. It combines the cached geoid-corrected PIXC profile, RiverSP reach/node values, updated filter result, and reach-year quantile-slope characterization into one publication figure.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

INK = "#17222B"
MUTED = "#647078"
GRID = "#DCE2E5"
BLUE = "#1976A8"
DARK_BLUE = "#084C7B"
RED = "#C83E4D"
TEAL = "#278F86"

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.edgecolor": INK,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "figure.dpi": 160,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

In [ ]:
def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "raqw").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root()
REACH_ID = "66405900761"
YEAR = 2024
FILE_NAME = "SWOT_L2_HR_PIXC_012_104_182L_20240310T103854_20240310T103905_PGD0_01.nc"

KEY_DIR = PROJECT_ROOT / "data" / "key_figures"
FILTER_DIR = PROJECT_ROOT / "data" / "tau_range_update_experiment" / f"reach_{REACH_ID}" / str(YEAR)
PROCESSED_DIR = PROJECT_ROOT / "data" / "chile_reaches_with_valid_discharge_run" / "processed" / f"reach_{REACH_ID}" / str(YEAR)
COMPARISON_DIR = PROJECT_ROOT / "data" / "riversp_comparison_updated_filter"

POINTS_CSV = KEY_DIR / "reach_66405900761_2024-03-10_clipped_geoid_points.csv"
NODES_CSV = KEY_DIR / "reach_66405900761_2024-03-10_hydrocron_nodes.csv"
UPDATED_CSV = FILTER_DIR / "updated_tau_window_results.csv"
BAND_CSV = FILTER_DIR / "universal_band.csv"
SLOPES_CSV = PROCESSED_DIR / "quantile_slopes_long.csv"
MATCHED_CSV = COMPARISON_DIR / "matched_pixc_riversp_slopes.csv"
RIVERSP_CACHE_CSV = COMPARISON_DIR / "riversp_daily_cache.csv"

FIGURE_DIR = PROJECT_ROOT / "publication_figs" / "outputs"
OUTPUT_STEM = "figure_02_example_reach_date_filter"
EXPORT_FILES = False
if EXPORT_FILES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

for path in [POINTS_CSV, NODES_CSV, UPDATED_CSV, BAND_CSV, SLOPES_CSV, MATCHED_CSV, RIVERSP_CACHE_CSV]:
    assert path.exists(), path

In [ ]:
points = pd.read_csv(POINTS_CSV).dropna(
    subset=["s_m", "height_ellipsoid_tide_corrected", "height_geoid_tide_corrected"]
).sort_values("s_m")
nodes = pd.read_csv(NODES_CSV)
nodes = nodes.loc[pd.to_numeric(nodes["wse"], errors="coerce").notna()].copy()
nodes["wse"] = pd.to_numeric(nodes["wse"], errors="coerce")
nodes = nodes.loc[nodes["wse"] > -1e10]
if "node_q" in nodes:
    nodes = nodes.loc[pd.to_numeric(nodes["node_q"], errors="coerce").fillna(9) <= 1]

updated = pd.read_csv(UPDATED_CSV)
updated_row = updated.loc[updated["file"] == FILE_NAME].iloc[0]
band = pd.read_csv(BAND_CSV).sort_values("quantile")
slopes = pd.read_csv(SLOPES_CSV)
file_curve = slopes.loc[slopes["file"] == FILE_NAME].sort_values("quantile")

matched = pd.read_csv(MATCHED_CSV, dtype={"reach_id": str})
matched_row = matched.loc[
    (matched["reach_id"] == REACH_ID)
    & (matched["year"].astype(int) == YEAR)
    & (matched["file"] == FILE_NAME)
].iloc[0]
cache = pd.read_csv(RIVERSP_CACHE_CSV, dtype={"reach_id": str})
riversp_row = cache.loc[
    (cache["reach_id"] == REACH_ID)
    & (cache["time_utc"] == matched_row["riversp_time_utc"])
].iloc[0]

x_m = points["s_m"].to_numpy(float)
x_km = x_m / 1000.0
y_ellipsoid = points["height_ellipsoid_tide_corrected"].to_numpy(float)
y_geoid = points["height_geoid_tide_corrected"].to_numpy(float)

detrend_slope = float(updated_row["updated_detrend_slope_m_per_km"]) / 1000.0
detrended = y_ellipsoid - detrend_slope * x_m
tau_low = float(updated_row["updated_tau_low"])
tau_high = float(updated_row["updated_tau_high"])
lower_height = np.quantile(detrended, tau_low)
upper_height = np.quantile(detrended, tau_high)
keep = (detrended >= lower_height) & (detrended <= upper_height)

raw_slope, raw_intercept = np.polyfit(x_m, y_geoid, 1)
filtered_slope, filtered_intercept = np.polyfit(x_m[keep], y_geoid[keep], 1)
raw_slope_km = raw_slope * 1000.0
filtered_slope_km = filtered_slope * 1000.0

line_x_m = np.linspace(x_m.min(), x_m.max(), 400)
line_x_km = line_x_m / 1000.0
raw_line = raw_intercept + raw_slope * line_x_m
filtered_line = filtered_intercept + filtered_slope * line_x_m

riversp_wse = float(riversp_row["wse"])
riversp_slope = float(riversp_row["riversp_slope"])
riversp_slope_km = riversp_slope * 1000.0
riversp_line = riversp_wse + riversp_slope * (line_x_m - x_m.mean())

curve = file_curve.merge(band, on="quantile", how="left")

print(f"Raw points: {len(points):,}")
print(f"Retained points: {keep.sum():,} ({keep.mean():.1%})")
print(f"Selected quantile range: {tau_low:.2f}-{tau_high:.2f}")
print(f"Raw geoid-corrected OLS slope: {raw_slope_km:.3f} m/km")
print(f"Filtered geoid-corrected OLS slope: {filtered_slope_km:.3f} m/km")
print(f"RiverSP slope: {riversp_slope_km:.3f} m/km")

In [ ]:
def panel_heading(ax, letter, title):
    ax.text(
        0, 1.025, f"{letter}   {title}", transform=ax.transAxes,
        ha="left", va="bottom", fontsize=10.5, fontweight="bold", color=INK,
    )


def style_profile_axis(ax):
    ax.grid(color=GRID, linewidth=0.55)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel("Distance along reach (km)")


fig = plt.figure(figsize=(11.0, 6.9), facecolor="white")
grid = fig.add_gridspec(
    2, 2, height_ratios=[1.32, 0.68],
    left=0.075, right=0.985, bottom=0.09, top=0.95,
    wspace=0.16, hspace=0.34,
)
ax_raw = fig.add_subplot(grid[0, 0])
ax_filtered = fig.add_subplot(grid[0, 1], sharex=ax_raw, sharey=ax_raw)
ax_tau = fig.add_subplot(grid[1, :])

# Panel A: raw profile and raw OLS.
ax_raw.scatter(
    x_km, y_geoid, s=7, color="#7D878C", alpha=0.24,
    linewidths=0, rasterized=True, label="Raw PIXC WSE",
)
ax_raw.plot(
    line_x_km, riversp_line, color=RED, linewidth=1.35,
    linestyle=(0, (5, 3)), label="RiverSP slope reference",
)
ax_raw.scatter(
    nodes["node_s_m"] / 1000.0, nodes["wse"],
    s=15, color=RED, edgecolor="white", linewidth=0.4, alpha=0.88,
    zorder=5, label="RiverSP node WSE",
)
ax_raw.text(
    0.03, 0.96,
    f"RiverSP: {riversp_slope_km:.3f} m km$^{{-1}}$",
    transform=ax_raw.transAxes, ha="left", va="top", fontsize=8,
    bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "#C8CED1", "alpha": 0.94},
)
ax_raw.set_ylabel("Geoid-referenced WSE (m)")
raw_legend = ax_raw.legend(loc="lower right", frameon=False, fontsize=7.3, markerscale=1.4)
for handle in raw_legend.legend_handles:
    handle.set_alpha(1)
style_profile_axis(ax_raw)
panel_heading(ax_raw, "a", "Raw PIXC WSE profile")

# Panel B: selected and rejected points with final fit.
ax_filtered.scatter(
    x_km[~keep], y_geoid[~keep], s=6, marker="x",
    color="#A6ADB1", alpha=0.80, linewidths=0.35, rasterized=True,
    label=f"Rejected (n={(~keep).sum():,})",
)
ax_filtered.scatter(
    x_km[keep], y_geoid[keep], s=9, color=BLUE, alpha=0.78,
    linewidths=0, rasterized=True, label=f"Retained (n={keep.sum():,})",
)
ax_filtered.plot(line_x_km, filtered_line, color=DARK_BLUE, linewidth=2.2, label="Filtered OLS")

ax_filtered.text(
    0.03, 0.96,
    f"Filtered OLS: {filtered_slope_km:.3f} m km$^{{-1}}$\n"
    f"Raw OLS: {raw_slope_km:.3f} m km$^{{-1}}$\n"
    f"Retained: {keep.mean():.0%}",
    transform=ax_filtered.transAxes, ha="left", va="top", fontsize=8,
    bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "#C8CED1", "alpha": 0.94},
)
filtered_legend = ax_filtered.legend(loc="lower right", frameon=False, fontsize=7.3, markerscale=1.1)
for handle in filtered_legend.legend_handles:
    handle.set_alpha(1)
style_profile_axis(ax_filtered)
panel_heading(ax_filtered, "b", "Filtered PIXC WSE profile")

# Use identical profile limits so the visual change is not caused by rescaling.
y_pad = 2.0
ax_raw.set_xlim(x_km.min() - 0.15, x_km.max() + 0.15)
ax_raw.set_ylim(y_geoid.min() - y_pad, y_geoid.max() + y_pad)

# Panel C: statistical basis for the selected quantile range.
ax_tau.axvspan(0.01, tau_low, color="#E9ECEE", alpha=0.55, linewidth=0)
ax_tau.axvspan(tau_low, tau_high, color="#BFE3DB", alpha=0.34, linewidth=0, label=f"Selected range ({tau_low:.2f}-{tau_high:.2f})")
ax_tau.axvspan(tau_high, 0.99, color="#E9ECEE", alpha=0.55, linewidth=0)
ax_tau.fill_between(
    curve["quantile"].to_numpy(float), curve["lower"].to_numpy(float), curve["upper"].to_numpy(float),
    color="#AEB8BD", alpha=0.24, linewidth=0, label="Reach-year robust band",
)
ax_tau.plot(curve["quantile"], curve["median"], color="#4E5A61", linewidth=1.6, label="Reach-year median")
ax_tau.plot(curve["quantile"], curve["slope_m_per_km"], color=BLUE, linewidth=2.0, label="Reach-date curve")
ax_tau.axvline(tau_low, color=TEAL, linewidth=0.9, linestyle="--")
ax_tau.axvline(tau_high, color=TEAL, linewidth=0.9, linestyle="--")
ax_tau.set_xlim(0.01, 0.99)
ax_tau.set_xlabel("Quantile, $\\tau$")
ax_tau.set_ylabel("Estimated slope (m km$^{-1}$)")
ax_tau.grid(color=GRID, linewidth=0.55)
ax_tau.set_axisbelow(True)
ax_tau.spines[["top", "right"]].set_visible(False)
tau_legend_handles = [
    mpatches.Patch(facecolor="#BFE3DB", edgecolor="none", alpha=0.45, label=f"Selected range ({tau_low:.2f}-{tau_high:.2f})"),
    mpatches.Patch(facecolor="#AEB8BD", edgecolor="none", alpha=0.28, label="Reach-year robust band"),
    mlines.Line2D([], [], color="#4E5A61", linewidth=1.8, label="Reach-year median"),
    mlines.Line2D([], [], color=BLUE, linewidth=2.2, label="Reach-date curve"),
]
tau_legend = ax_tau.legend(
    handles=tau_legend_handles,
    loc="upper right", bbox_to_anchor=(1.0, 1.20), ncol=4,
    frameon=True, fontsize=7.3, columnspacing=1.15, handlelength=2.0,
    borderpad=0.35, labelspacing=0.55,
)
tau_legend.get_frame().set_facecolor("white")
tau_legend.get_frame().set_edgecolor("#D5DCDD")
tau_legend.get_frame().set_linewidth(0.6)
tau_legend.get_frame().set_alpha(0.96)
panel_heading(ax_tau, "c", "Quantile-slope curve and selected range")

if EXPORT_FILES:
    png_path = FIGURE_DIR / f"{OUTPUT_STEM}.png"
    pdf_path = FIGURE_DIR / f"{OUTPUT_STEM}.pdf"
    fig.savefig(png_path, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    print(f"Saved {png_path}")
    print(f"Saved {pdf_path}")

plt.show()

## Draft caption

**Figure 2. Representative example of PIXC-level filtering and slope estimation for one reach-overpass.** (a) Raw PIXC WSE profile as a function of downstream distance, shown with RiverSP node WSE values and the RiverSP reach-scale slope reference line. (b) Filtered PIXC profile after applying the selected quantile range; retained points are used in the final OLS slope estimate, rejected points are shown only for reference, and the filtered OLS fit is shown as a solid line. (c) Quantile-slope behavior used to identify the stable tau range. This example illustrates how the workflow removes slope-relevant outliers while preserving the retained PIXC point locations used to estimate reach-scale slope.